# Build derived TorNet modeling manifests

Build a validated modeling manifest from the preserved raw
annual audit without rescanning or modifying the source archive.

The official test set is preserved exactly. Explicitly documented
training exclusions resolve confirmed train/test event leakage.


In [1]:
%pip install -q xarray netCDF4 pandas pyarrow


In [2]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%pip install --force-reinstall --no-deps "/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.2-py3-none-any.whl"


Processing ./drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.2-py3-none-any.whl
  Attempting uninstall: tornet-detection
    Found existing installation: tornet-detection 0.1.1
    Uninstalling tornet-detection-0.1.1:
      Successfully uninstalled tornet-detection-0.1.1


In [4]:
from pathlib import Path

import pandas as pd

import tornado_detection

from tornado_detection.data.modeling import (
    ModelingExclusion,
    build_modeling_manifest,
    write_modeling_manifest_artifacts,
)

assert tornado_detection.__version__ == "0.1.2"

print(
    "tornado_detection package version:",
    tornado_detection.__version__,
)
print(
    "Loaded tornado_detection from:",
    tornado_detection.__file__,
)


tornado_detection package version: 0.1.2
Loaded tornado_detection from: /usr/local/lib/python3.12/dist-packages/tornado_detection/__init__.py


## 2014 leakage resolution


In [5]:
RAW_DIRECTORY = Path(
    "/content/drive/MyDrive/TorNet_Backup/manifests/v1/2014"
)

OUTPUT_DIRECTORY = Path(
    "/content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2014"
)

EXPECTED_DIMENSIONS = {
    "time": 4,
    "sweep": 2,
    "azimuth": 120,
    "range": 240,
    "lims": 2,
}

EXCLUSIONS = [
    ModelingExclusion(
        archive_member=(
            "train/2014/"
            "TOR_140429_234928_KRAX_505771_A8.nc"
        ),
        expected_event_id="505771",
        expected_episode_id="83781",
        reason=(
            "Official train/test event leakage"
        ),
        resolution=(
            "Preserve official test files and exclude "
            "the overlapping training member"
        ),
    )
]

print("Raw manifest:", RAW_DIRECTORY)
print("Modeling output:", OUTPUT_DIRECTORY)


Raw manifest: /content/drive/MyDrive/TorNet_Backup/manifests/v1/2014
Modeling output: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2014


In [6]:
modeling_build = build_modeling_manifest(
    RAW_DIRECTORY,
    exclusions=EXCLUSIONS,
    expected_dimensions=EXPECTED_DIMENSIONS,
)

display(modeling_build.exclusion_ledger)
display(modeling_build.validation.checks)
display(
    modeling_build.validation
    .category_frame_summary
)

assert len(
    modeling_build.manifest.file_manifest
) == 19_992

assert len(
    modeling_build.manifest.frame_manifest
) == 79_968

assert int(
    modeling_build.exclusion_ledger[
        "removed_positive_frame_count"
    ].sum()
) == 1

assert (
    modeling_build.validation
    .event_split_overlap.empty
)

assert (
    modeling_build.validation
    .all_required_passed
)

print(
    "PASS: 2014 modeling manifest validated "
    "before writing"
)


,manifest_schema_version,year,archive_member,file_id,split,category,event_id,episode_id,radar_site,removed_frame_count,removed_positive_frame_count,reason,resolution
0,1.0.0,2014,train/2014/TOR_140429_234928_KRAX_505771_A8.nc,660a6c9b1ae4b817936ea3f6439c2d0a274b6db4c12262...,train,TOR,505771,83781,KRAX,4,1,Official train/test event leakage,Preserve official test files and exclude the o...


,check,required,passed,observed,expected,detail
0,build_errors,True,True,0,0,
1,file_row_count,True,True,19992,19992,
2,frame_row_count,True,True,79968,79968,
3,unique_archive_members,True,True,19992,19992,
4,unique_file_ids,True,True,19992,19992,
5,unique_frame_ids,True,True,79968,79968,
6,frame_rows_match_file_frame_counts,True,True,0,0,
7,frame_label_sums_match_file_manifest,True,True,0,0,
8,frame_indices_are_contiguous,True,True,0,0,
9,expected_frames_per_file,True,True,0,0,Expected 4 frames for every file


,split,category,file_count,frame_count,positive_frame_count,files_with_positive_frames,files_with_mixed_frame_labels,positive_frame_prevalence
0,test,NUL,1345,5380,0,0,0,0.000000
1,test,TOR,353,1412,792,353,278,0.560907
2,test,WRN,848,3392,0,0,0,0.000000
3,train,NUL,9934,39736,0,0,0,0.000000
4,train,TOR,1175,4700,2259,1175,1035,0.480638
5,train,WRN,6337,25348,0,0,0,0.000000


PASS: 2014 modeling manifest validated before writing


In [7]:
artifacts = write_modeling_manifest_artifacts(
    modeling_build,
    OUTPUT_DIRECTORY,
    overwrite=False,
)

for artifact_name, artifact_path in (
    artifacts.items()
):
    print(
        f"- {artifact_name}: "
        f"{artifact_path}"
    )

print(
    "PASS: 2014 modeling manifest built "
    "and written"
)


- build_errors.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2014/build_errors.csv
- category_frame_summary.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2014/category_frame_summary.csv
- episode_split_overlap.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2014/episode_split_overlap.csv
- event_split_overlap.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2014/event_split_overlap.csv
- exclusion_ledger.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2014/exclusion_ledger.csv
- file_manifest.parquet: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2014/file_manifest.parquet
- frame_manifest.parquet: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2014/frame_manifest.parquet
- manifest_summary.json: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2014/manifest_summary.json
- modeling_summary.json: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2014/modeling_su